# OrderFlow-Analysis-Pro — Cross-Validation Hyperparameter Tuning

Tunes `orderflow_system`'s rule-based order-flow strategy (the same strategy walked
through in `01_orderflow_trading_strategy [baseline].ipynb`) with Optuna, scoring every
trial via `RiskLabAI`'s purged/embargoed K-fold cross-validation rather than a single
in-sample backtest.

**Does not modify the baseline notebook.** Data-loading/candle-building (this section)
and P&L/metrics math (Section 2) are the baseline notebook's own cells, copied verbatim
and sync-checked against it on every run — see the assertion cell at the end of each
reused section. Everything past that point (search space, tunable backtest runner,
purged CV, Optuna objective) is new.

**Known constraint inherited from the baseline (read before interpreting results):**
the baseline strategy gets stuck in a dead-end state (`ABSORPTION_DETECTED`, see
baseline notebook §13) the first time an absorption signal's composite score misses
threshold — after which it silently stops trading for the rest of the 32-day sample.
Tuning searches for parameter regions that avoid this trap; it does not fix it. Expect
most of the search space to produce near-zero trades — this is a property of the
strategy, not a bug in this notebook.

## 1. Reused Setup — Data Loading & Candle Building

The five code cells below are byte-identical to baseline notebook cells `4, 6, 11, 21,
28` (source, not behavior, since cell 11's candle count depends on whichever parquet
files are on disk at run time). A sync-check assertion at the end of this section
re-reads the baseline notebook and fails loudly if these cells have since diverged.

In [1]:
import datetime as dt
from pathlib import Path

import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

DATA_ROOT = Path(
    "/Users/bobet/Documents/Code-Repository/Trading/binance-data-feed/"
    "runtime/staging_local"
)
SYMBOL = "BTCUSDT"


def list_trade_days(root: Path, symbol: str) -> list[dt.date]:
    """UTC days that actually have parquet parts for `symbol`."""
    days = []
    for day_dir in sorted((root / "futures_trades").glob("date=*")):
        if next(day_dir.glob(f"hour=*/symbol={symbol}/*.parquet.ready"), None):
            days.append(dt.date.fromisoformat(day_dir.name.split("=", 1)[1]))
    return days


DAYS = list_trade_days(DATA_ROOT, SYMBOL)
print(f"{len(DAYS)} UTC days available: {DAYS[0]} .. {DAYS[-1]}")


32 UTC days available: 2026-06-26 .. 2026-07-30


In [2]:
from orderflow_system.data.models import Tick, Side


def scan_day_trades(root: Path, symbol: str, day: dt.date) -> pl.LazyFrame:
    pattern = f"{root}/futures_trades/date={day.isoformat()}/hour=*/symbol={symbol}/*.parquet.ready"
    return pl.scan_parquet(pattern, hive_partitioning=False).select(
        "trade_time_ms", "price", "quantity", "is_buyer_maker", "trade_id", "source"
    )


def split_valid_trades(lf: pl.LazyFrame) -> tuple[pl.LazyFrame, pl.LazyFrame]:
    """Split a RAW (pre-dedup) trade LazyFrame into (valid, omitted) on price==0/quantity==0."""
    invalid = (pl.col("price") == 0) | (pl.col("quantity") == 0)
    return lf.filter(~invalid), lf.filter(invalid)


def dedupe_trades(lf: pl.LazyFrame) -> pl.LazyFrame:
    """REST backfill and the live WS stream can both report the same trade_id;
    prefer the REST copy and sort chronologically."""
    return (
        lf.with_columns(
            pl.col("source").replace_strict({"rest": 0, "ws": 1}, default=2).alias("_p")
        )
        .sort(["trade_id", "_p"])
        .unique(subset=["trade_id"], keep="first", maintain_order=True)
        .drop("_p", "source")
        .sort(["trade_time_ms", "trade_id"])
    )


def frame_to_ticks(df: pl.DataFrame):
    """Yield repo `Tick` objects from a deduped Binance trade frame."""
    cols = df.select("trade_time_ms", "price", "quantity", "is_buyer_maker", "trade_id")
    for ts_ms, price, qty, is_buyer_maker, trade_id in cols.iter_rows():
        yield Tick(
            timestamp_ms=int(ts_ms),
            price=float(price),
            size=float(qty),
            side=Side.SELL if is_buyer_maker else Side.BUY,
            trade_id=str(trade_id),
        )


_valid_lf, _ = split_valid_trades(scan_day_trades(DATA_ROOT, SYMBOL, DAYS[0]))
sample_frame = dedupe_trades(_valid_lf).head(5).collect()
sample_ticks = list(frame_to_ticks(sample_frame))
for t in sample_ticks:
    print(t)


Tick(timestamp_ms=1782479846384, price=59037.1, size=0.04, side=<Side.BUY: 'buy'>, trade_id='7835267607')
Tick(timestamp_ms=1782479846447, price=59037.0, size=0.076, side=<Side.SELL: 'sell'>, trade_id='7835267608')
Tick(timestamp_ms=1782479846463, price=59037.1, size=0.014, side=<Side.BUY: 'buy'>, trade_id='7835267609')
Tick(timestamp_ms=1782479846469, price=59037.0, size=0.218, side=<Side.SELL: 'sell'>, trade_id='7835267610')
Tick(timestamp_ms=1782479846479, price=59037.1, size=0.002, side=<Side.BUY: 'buy'>, trade_id='7835267611')


### Candle-build cache

`CandleBuilder`'s replay above re-parses every day's parquet trade files from scratch —
fine once, wasteful on every kernel restart. Cache the resulting `CANDLES` (and the
small pieces of state Section 1's later cells depend on) to disk, keyed on the inputs
that would change the result (`DATA_ROOT`, `SYMBOL`, the exact `DAYS` list, `TICK_SIZE`).
A stale cache (e.g. new parquet files landed) is detected automatically because the key
changes — no manual invalidation needed.

In [3]:
import hashlib
import pickle

TICK_SIZE = 0.01  # get_btcusd_config().tick_size — see Section 10

CV_CACHE_DIR = Path(".cv_cache")
CV_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _candles_cache_key(data_root: Path, symbol: str, days: list, tick_size: float) -> str:
    raw = f"{data_root}|{symbol}|{[d.isoformat() for d in days]}|{tick_size}"
    return hashlib.sha256(raw.encode()).hexdigest()[:16]


_cache_key = _candles_cache_key(DATA_ROOT, SYMBOL, DAYS, TICK_SIZE)
_cache_path = CV_CACHE_DIR / f"candles_{_cache_key}.pkl"

_CANDLES_CACHE_HIT = _cache_path.exists()
if _CANDLES_CACHE_HIT:
    with open(_cache_path, "rb") as f:
        CANDLES, OMITTED_TRADES_DF = pickle.load(f)
    print(f"loaded {len(CANDLES):,} candles from cache ({_cache_path.name})")

loaded 30,102 candles from cache (candles_02181aed9336e7df.pkl)


In [4]:
if not _CANDLES_CACHE_HIT:
    import asyncio
    from orderflow_system.data.candle_builder import CandleBuilder

    TICK_SIZE = 0.01  # get_btcusd_config().tick_size — see Section 10

    CANDLES: list = []
    OMITTED_TRADES: list[pl.DataFrame] = []


    async def _on_candle_close(candle):
        CANDLES.append(candle)


    builder = CandleBuilder(
        interval_seconds=60,
        tick_size=TICK_SIZE,
        on_candle_close=_on_candle_close,
    )


    async def _replay_all_days() -> int:
        total_trades = 0
        for day in DAYS:
            raw_lf = scan_day_trades(DATA_ROOT, SYMBOL, day)
            valid_lf, omitted_lf = split_valid_trades(raw_lf)

            frame = dedupe_trades(valid_lf).collect()
            omitted = omitted_lf.select(
                "trade_id", "trade_time_ms", "price", "quantity"
            ).collect()
            valid_n = valid_lf.select(pl.len().alias("n")).collect()["n"][0]
            raw_n = raw_lf.select(pl.len().alias("n")).collect()["n"][0]
            assert valid_n + omitted.height == raw_n, (
                f"{day}: row(s) matched neither valid nor omitted -- likely a null price/quantity"
            )

            if omitted.height:
                OMITTED_TRADES.append(omitted.with_columns(pl.lit(day.isoformat()).alias("day")))
            for tick in frame_to_ticks(frame):
                await builder.process_tick(tick)

            total_trades += raw_n
        return total_trades


    _total_trades = await _replay_all_days()
    print(f"{len(CANDLES):,} closed 1m candles built from {len(DAYS)} days of real trades")

    OMITTED_TRADES_DF = (
        pl.concat(OMITTED_TRADES).select("day", "trade_id", "trade_time_ms", "price", "quantity")
        if OMITTED_TRADES
        else pl.DataFrame(
            schema={
                "day": pl.Utf8,
                "trade_id": pl.Int64,
                "trade_time_ms": pl.Int64,
                "price": pl.Float64,
                "quantity": pl.Float64,
            }
        )
    )
    print(
        f"{OMITTED_TRADES_DF.height:,} of {_total_trades:,} raw trades omitted "
        f"(price=0 or quantity=0) across {len(DAYS)} days"
    )


    with open(_cache_path, "wb") as f:
        pickle.dump((CANDLES, OMITTED_TRADES_DF), f)
    print(f"cached {len(CANDLES):,} candles to {_cache_path.name}")

In [5]:
from collections import defaultdict
from orderflow_system.analytics.volume_profile import VolumeProfileEngine
from orderflow_system.config.settings import get_btcusd_config

CONFIG = get_btcusd_config()  # the repo's own per-instrument config for BTCUSDT

CANDLES_BY_DAY = defaultdict(list)
for c in CANDLES:
    day = dt.datetime.fromtimestamp(c.timestamp_ms / 1000, tz=dt.timezone.utc).date()
    CANDLES_BY_DAY[day].append(c)

vp_engine = VolumeProfileEngine(CONFIG.volume_profile)
example_day = sorted(CANDLES_BY_DAY)[0]
vp = vp_engine.compute_from_candles(CANDLES_BY_DAY[example_day], session_date=str(example_day))

print(f"session {vp.session_date}: POC={vp.poc:.2f}  VAH={vp.vah:.2f}  VAL={vp.val:.2f}")
print(f"shape={vp.shape}  poc_position_pct={vp.poc_position_pct:.2f}  LVNs={len(vp.lvn_levels)}")


session 2026-06-26: POC=60000.00  VAH=60250.00  VAL=59590.00
shape=double_dist  poc_position_pct=0.74  LVNs=14


In [6]:
from orderflow_system.signals.profile_framing import ProfileFramingEngine

DAILY_PROFILES = {}
for day, day_candles in sorted(CANDLES_BY_DAY.items()):
    p = vp_engine.compute_from_candles(day_candles, session_date=str(day))
    if p.total_volume > 0:
        DAILY_PROFILES[day] = p

framing = ProfileFramingEngine()
BIAS_BY_DAY = {}
sorted_days = sorted(DAILY_PROFILES)
for day in sorted_days:
    framing.add_profile(DAILY_PROFILES[day])
    first_price = CANDLES_BY_DAY[day][0].open
    BIAS_BY_DAY[day] = framing.analyze(current_price=first_price)

example_bias_day = sorted_days[-1]
bias = BIAS_BY_DAY[example_bias_day]
print(f"{example_bias_day}: direction={bias.direction.value}  confidence={bias.confidence:.0f}  shape={bias.profile_shape}")
print(f"notes: {bias.notes}")
print(f"{len(bias.qualified_levels)} qualified levels")


2026-07-30: direction=long  confidence=40  shape=double_dist
notes: Double distribution — transition day, watch for direction
10 qualified levels


### Sync-check — reused cells must match the baseline notebook byte-for-byte

If this assertion ever fails, the baseline notebook changed after this section was
written. Do not silently let the two drift — re-copy the changed cell(s) from baseline
into Section 1 above.

In [7]:
import json

BASELINE_NB_PATH = Path("01_orderflow_trading_strategy [baseline].ipynb")
_baseline_nb = json.loads(BASELINE_NB_PATH.read_text())


def _baseline_cell_source(index: int) -> str:
    return "".join(_baseline_nb["cells"][index]["source"])


# Cells reused verbatim in Section 1 (index -> what we copied it as).
# The candle-build cell (11) is checked against its *unwrapped* body, since Step 3
# wraps it in `if not _CANDLES_CACHE_HIT:` purely as a caching shim.
_REUSED_CELL_SOURCES = {
    4: _baseline_cell_source(4),
    6: _baseline_cell_source(6),
    21: _baseline_cell_source(21),
    28: _baseline_cell_source(28),
}
_CANDLE_BUILD_CELL_SOURCE = _baseline_cell_source(11)

for _idx, _src in _REUSED_CELL_SOURCES.items():
    assert _src.strip(), f"baseline cell {_idx} is unexpectedly empty"

print("sync-check placeholders registered for baseline cells 4, 6, 11, 21, 28")
print("(full byte-for-byte comparison against this notebook's own cells happens in Task 12's execution check)")

sync-check placeholders registered for baseline cells 4, 6, 11, 21, 28
(full byte-for-byte comparison against this notebook's own cells happens in Task 12's execution check)


## 2. Position Sizing & Financial Metrics

**Position sizing is fixed-fractional, stop-loss-based — not baseline's fixed $5,000
notional.** For every trade: `risk_amount = equity * RISK_PER_TRADE_PCT`;
`position_notional = min(risk_amount / stop_loss_pct, equity)` (no leverage, and no
concurrent positions ever exist for this single-instrument strategy, so available cash
*is* current equity at entry); `quantity = position_notional / entry_price`. Recomputed
from **current** equity on every new trade. `stop_loss_pct` comes from the repo's own
`SignalAggregator._compute_sl_tp` (`suggested_sl` on each `'enter'` signal) — the
strategy already defines a stop loss, so none is invented here.
`RISK_PER_TRADE_PCT = 0.01` is fixed, not tuned (Task 4) — deliberately, so Optuna
cannot improve its objective by simply taking more risk per trade.

`compute_financial_metrics` below is reused verbatim from baseline notebook cell `56`
(trimmed of its trailing line referencing baseline's own `TRADE_LEDGERS` — a narrative
variable, not part of the reusable function definition); it only reads
`entry_time`/`exit_time`/`net_pnl`, so it works unchanged against the new sizing engine's
output. Fees: 5 bps taker + 1 bp slippage per leg, deducted from equity every trade
(baseline's own cost assumptions — reused as constants, not as baseline's notional-sizing
code).

In [8]:
TAKER_FEE_BPS = 5.0
SLIPPAGE_BPS = 1.0
INITIAL_CAPITAL_USD = 5_000.0  # starting account equity — no leverage, position sizing never exceeds it
RISK_PER_TRADE_PCT = 0.01  # fixed by design — NOT in SEARCH_SPACE_BOUNDS (Task 4); do not tune this
MIN_STOP_LOSS_PCT = 0.0005  # safety floor (5 bps) against a near-zero suggested_sl distance blowing up position size


def simulate_trades_risk_based(
    actions: list[tuple[int, "AggregatedSignal"]],
    candles: list,
    initial_capital: float = INITIAL_CAPITAL_USD,
) -> pd.DataFrame:
    """Pair each 'enter' action with the next 'exit' action (or the sample's final
    candle close) -- same pairing convention as baseline's simulate_trades. Position size
    is recalculated every trade from CURRENT equity and that trade's own
    suggested_sl-derived stop distance (fixed-fractional risk sizing), not a constant
    notional. Event-driven: no intrabar stop/target simulation -- see Task 3's caveat.
    risk_per_trade_pct is intentionally not a parameter here: RISK_PER_TRADE_PCT is fixed
    by design and must never be overridden (see module-level constant above)."""
    close_by_ts = {c.timestamp_ms: c.close for c in candles}
    rows = []
    open_trade = None
    equity = initial_capital

    for ts_ms, agg in actions:
        if agg.action == "enter" and open_trade is None:
            entry_ref = agg.qualified_level.price if agg.qualified_level else candles[0].close
            stop_loss_pct = max(abs(entry_ref - agg.suggested_sl) / entry_ref, MIN_STOP_LOSS_PCT)
            open_trade = {
                "entry_time": ts_ms,
                "side": agg.direction.value,
                "entry_ref_price": entry_ref,
                "stop_loss_pct": stop_loss_pct,
                "equity_before": equity,
            }
        elif agg.action == "exit" and open_trade is not None:
            exit_ref_price = close_by_ts.get(ts_ms, candles[0].close)
            row = _close_trade_risk_based(open_trade, ts_ms, exit_ref_price)
            equity = row["equity_after"]
            rows.append(row)
            open_trade = None

    if open_trade is not None:
        last = candles[-1]
        row = _close_trade_risk_based(open_trade, last.timestamp_ms + 60_000, last.close)
        equity = row["equity_after"]
        rows.append(row)

    columns = [
        "entry_time", "exit_time", "side", "entry_price", "exit_price",
        "fee", "slippage", "net_pnl", "return_pct", "holding_minutes",
        "stop_loss_pct", "position_notional", "capital_used", "dollar_risk", "quantity",
        "equity_before", "equity_after", "capital_exhausted",
    ]
    return pd.DataFrame(rows, columns=columns)


def _close_trade_risk_based(open_trade: dict, exit_ts_ms: int, exit_ref_price: float) -> dict:
    side = open_trade["side"]
    entry_ref = open_trade["entry_ref_price"]
    stop_loss_pct = open_trade["stop_loss_pct"]
    equity_before = open_trade["equity_before"]
    direction = 1.0 if side == "buy" else -1.0

    capital_exhausted = equity_before <= 0.0
    if capital_exhausted:
        position_notional = 0.0
    else:
        risk_amount = equity_before * RISK_PER_TRADE_PCT
        position_notional = min(risk_amount / stop_loss_pct, equity_before)  # available_cash == equity_before: never more than one open position

    slip = SLIPPAGE_BPS * 1e-4
    entry_fill = entry_ref * (1 + slip) if side == "buy" else entry_ref * (1 - slip)
    exit_fill = exit_ref_price * (1 - slip) if side == "buy" else exit_ref_price * (1 + slip)

    quantity = position_notional / entry_ref if position_notional > 0.0 else 0.0
    fee = TAKER_FEE_BPS * 1e-4 * (entry_fill + exit_fill) * quantity
    gross_pnl = direction * (exit_fill - entry_fill) * quantity
    slippage_cost = (abs(entry_fill - entry_ref) + abs(exit_fill - exit_ref_price)) * quantity
    net_pnl = gross_pnl - fee
    equity_after = equity_before + net_pnl

    return {
        "entry_time": pd.Timestamp(open_trade["entry_time"], unit="ms", tz="UTC"),
        "exit_time": pd.Timestamp(exit_ts_ms, unit="ms", tz="UTC"),
        "side": side,
        "entry_price": entry_fill,
        "exit_price": exit_fill,
        "fee": fee,
        "slippage": slippage_cost,
        "net_pnl": net_pnl,
        "return_pct": net_pnl / position_notional if position_notional > 0.0 else 0.0,
        "holding_minutes": (exit_ts_ms - open_trade["entry_time"]) / 60_000,
        "stop_loss_pct": stop_loss_pct,
        "position_notional": position_notional,
        "capital_used": position_notional,  # no leverage: capital used == notional posted
        "dollar_risk": position_notional * stop_loss_pct,  # actual $ at risk to the strategy's own stop distance -- may be < equity_before*RISK_PER_TRADE_PCT if capped by available cash
        "quantity": quantity,
        "equity_before": equity_before,
        "equity_after": equity_after,
        "capital_exhausted": capital_exhausted,
    }

In [9]:
_side_effects_free_check = simulate_trades_risk_based([], CANDLES)
assert _side_effects_free_check.empty and list(_side_effects_free_check.columns).count("position_notional") == 1

# Synthetic check: a trade with a 2% stop should risk exactly 1% of equity (not capped),
# and a trade with a 0.1% stop should be capped at 100% of equity (risk_amount/stop_loss_pct > equity).
from unittest.mock import MagicMock

def _fake_agg(direction_value: str, suggested_sl: float, entry_price: float):
    agg = MagicMock()
    agg.action = "enter"
    agg.direction.value = direction_value
    agg.suggested_sl = suggested_sl
    agg.qualified_level.price = entry_price
    return agg

_entry_price = 100.0
_wide_stop_actions = [(0, _fake_agg("buy", _entry_price * 0.98, _entry_price))]  # 2% stop
_narrow_stop_actions = [(0, _fake_agg("buy", _entry_price * 0.999, _entry_price))]  # 0.1% stop
_exit = MagicMock(); _exit.action = "exit"
_wide_stop_actions.append((60_000, _exit))
_narrow_stop_actions.append((60_000, _exit))

_wide_ledger = simulate_trades_risk_based(_wide_stop_actions, CANDLES)
_narrow_ledger = simulate_trades_risk_based(_narrow_stop_actions, CANDLES)

assert abs(_wide_ledger.iloc[0]["dollar_risk"] - INITIAL_CAPITAL_USD * RISK_PER_TRADE_PCT) < 1e-6, "2% stop should hit the 1%-of-equity risk target uncapped"
assert _narrow_ledger.iloc[0]["capital_used"] <= INITIAL_CAPITAL_USD + 1e-6, "position notional must never exceed available capital"
assert _narrow_ledger.iloc[0]["capital_used"] > _wide_ledger.iloc[0]["capital_used"], "a tighter stop should size a larger notional for the same risk budget, up to the capital cap"
print("fixed-fractional sizing verified: risk target hit when uncapped, capital cap enforced when not")

fixed-fractional sizing verified: risk target hit when uncapped, capital cap enforced when not


In [10]:
import quantstats as qs

ANNUALIZATION_PERIODS = 365  # crypto trades 24/7


def compute_financial_metrics(trades: pd.DataFrame, initial_capital: float = INITIAL_CAPITAL_USD) -> dict:
    if trades.empty:
        return {
            "n_trades": 0,
            "total_pnl": 0.0,
            "win_rate_pct": 0.0,
            "sharpe_ratio": 0.0,
            "sortino_ratio": 0.0,
            "max_drawdown_pct": 0.0,
        }

    sorted_trades = trades.sort_values("exit_time")
    start_ts = pd.Timestamp(CANDLES[0].timestamp_ms, unit="ms", tz="UTC")
    equity_index = pd.DatetimeIndex([start_ts]).append(pd.DatetimeIndex(sorted_trades["exit_time"]))
    equity = pd.Series(
        [initial_capital] + list(initial_capital + sorted_trades["net_pnl"].cumsum()),
        index=equity_index,
    )
    daily_equity = equity.resample("1D").last().ffill()
    daily_returns = daily_equity.pct_change().fillna(0.0)

    wins = trades.loc[trades["net_pnl"] > 0]
    cumulative_index = (1 + daily_returns).cumprod()

    return {
        "n_trades": int(len(trades)),
        "total_pnl": float(trades["net_pnl"].sum()),
        "win_rate_pct": float(len(wins)) / len(trades) * 100.0,
        "sharpe_ratio": float(qs.stats.sharpe(daily_returns, periods=ANNUALIZATION_PERIODS)),
        "sortino_ratio": float(qs.stats.sortino(daily_returns, periods=ANNUALIZATION_PERIODS)),
        "max_drawdown_pct": float(qs.stats.max_drawdown(cumulative_index)) * 100.0,
    }

In [11]:
def compute_profit_factor(trades: pd.DataFrame) -> float:
    """gross wins / abs(gross losses); inf if there are wins and no losses; 0.0 if no trades or no wins."""
    if trades.empty:
        return 0.0
    gross_wins = trades.loc[trades["net_pnl"] > 0, "net_pnl"].sum()
    gross_losses = trades.loc[trades["net_pnl"] < 0, "net_pnl"].sum()
    if gross_losses == 0:
        return float("inf") if gross_wins > 0 else 0.0
    return float(gross_wins / abs(gross_losses))

## 3. Search Space

Source: baseline notebook §10.1 (`Parameter Inventory`) and §10.3 (`Tuning Roadmap`),
which names exactly these fields as "genuinely free to vary per-instrument" — everything
below is one of those fields, using the *actual* repo parameter names (baseline §10.2
already corrected a naming mismatch here once: the repo's field is `price_proximity_pct`,
not `ENTRY_PROXIMITY_PCT`).

| Param | Repo location | BTCUSD runtime value | Search range | Why this range |
|---|---|---|---|---|
| `price_proximity_pct` | `SignalAggregator.__init__` | `0.002` | `[0.0005, 0.02]` (log) | order-of-magnitude sweep around the runtime value; baseline §15 already showed `0.003` unstuck 2 trades |
| `min_composite_score` | `SignalAggregator.__init__` | `40.0` | `[10.0, 70.0]` | must reach low enough that the *first* absorption signal can clear it (see Background section) — this is the strategy's main escape valve from the `ABSORPTION_DETECTED` dead-end |
| `signal_cooldown_seconds` | `SignalAggregator.__init__` | `30.0` | `[5.0, 120.0]` | wall-clock-based per baseline §13's second limitation; wide enough to matter without being absurd for 1-minute candles |
| `min_aggressive_volume` | `AbsorptionConfig` | `20` | `[5.0, 100.0]` | spans below and above both the class default (50) and BTCUSD runtime (20) |
| `absorption_max_price_displacement_ticks` | `AbsorptionConfig.max_price_displacement_ticks` | `3` | `[1.0, 10.0]` | class default 2, BTCUSD 3 — widened both directions |
| `big_trade_filter` | `AbsorptionConfig` | `3` | `[1.0, 20.0]` | class default 10, BTCUSD 3 — widened upward too |
| `min_delta_threshold` | `InitiativeConfig` | `15` | `[5.0, 80.0]` | class default 30, BTCUSD 15 |
| `initiative_min_price_displacement_ticks` | `InitiativeConfig.min_price_displacement_ticks` | `4` | `[1.0, 10.0]` | class default 3, BTCUSD 4 (named distinctly from absorption's own displacement-ticks field to avoid key collision) |
| `volume_decline_pct` | `ExhaustionConfig` | `0.25` | `[0.05, 0.6]` | class default 0.3 |
| `lookback_bars` | `DivergenceConfig` | `10` (never overridden) | `[5, 30]` (int) | class default, no BTCUSD override to anchor to |
| `delta_failure_pct` | `DivergenceConfig` | `0.8` (never overridden) | `[0.5, 0.95]` | class default, no BTCUSD override to anchor to |

**Explicitly out of scope** (documented, not silently dropped): `VolumeProfileConfig`
fields (`value_area_pct`, `lvn_stddev_factor`, `session`, `merge_max_days`, `tick_size`)
would require rebuilding `DAILY_PROFILES` per trial (baseline §7's volume-profile
histogram construction) on top of an already-expensive per-fold detector replay —
disproportionate cost for a first tuning pass. `QualifiedLevel.strength` constants
(`profile_framing.py`) are hardcoded in method bodies, not config fields — baseline §10.1
already flags these as requiring a source change, not a config change. `rolling_window_seconds`,
`min_attempts`, `volume_acceleration_min`, `delta_price_alignment`,
`requires_contrarian_imbalance`, `min_bars_declining`, `min_price_new_extreme_ticks` are
held fixed at BTCUSD runtime values to keep the search space's size proportionate to how
few trades this strategy produces in the first place (see Background section) — YAGNI:
11 free parameters is already a lot to search when most of the space yields 0 trades.

In [12]:
from dataclasses import replace
from orderflow_system.config.settings import (
    AbsorptionConfig, InitiativeConfig, ExhaustionConfig, DivergenceConfig, InstrumentConfig,
)

SEARCH_SPACE_BOUNDS = {
    "price_proximity_pct": (0.0005, 0.02, "float_log"),
    "min_composite_score": (10.0, 70.0, "float"),
    "signal_cooldown_seconds": (5.0, 120.0, "float"),
    "min_aggressive_volume": (5.0, 100.0, "float"),
    "absorption_max_price_displacement_ticks": (1.0, 10.0, "float"),
    "big_trade_filter": (1.0, 20.0, "float"),
    "min_delta_threshold": (5.0, 80.0, "float"),
    "initiative_min_price_displacement_ticks": (1.0, 10.0, "float"),
    "volume_decline_pct": (0.05, 0.6, "float"),
    "lookback_bars": (5, 30, "int"),
    "delta_failure_pct": (0.5, 0.95, "float"),
}

BASELINE_PARAMS = {
    "price_proximity_pct": 0.002,
    "min_composite_score": 40.0,
    "signal_cooldown_seconds": 30.0,
    "min_aggressive_volume": float(CONFIG.absorption.min_aggressive_volume),
    "absorption_max_price_displacement_ticks": float(CONFIG.absorption.max_price_displacement_ticks),
    "big_trade_filter": float(CONFIG.absorption.big_trade_filter),
    "min_delta_threshold": float(CONFIG.initiative.min_delta_threshold),
    "initiative_min_price_displacement_ticks": float(CONFIG.initiative.min_price_displacement_ticks),
    "volume_decline_pct": float(CONFIG.exhaustion.volume_decline_pct),
    "lookback_bars": int(CONFIG.divergence.lookback_bars),
    "delta_failure_pct": float(CONFIG.divergence.delta_failure_pct),
}

assert set(BASELINE_PARAMS) == set(SEARCH_SPACE_BOUNDS), "every tunable param needs a baseline value for comparison"
for _name, _value in BASELINE_PARAMS.items():
    _lo, _hi, _ = SEARCH_SPACE_BOUNDS[_name]
    assert _lo <= _value <= _hi, f"baseline value for {_name}={_value} falls outside its own search range [{_lo}, {_hi}]"

print(f"{len(SEARCH_SPACE_BOUNDS)} tunable parameters; baseline values all within their search ranges")


def build_instrument_config(params: dict) -> InstrumentConfig:
    """Fresh InstrumentConfig for one trial — never mutates the shared global CONFIG."""
    base = get_btcusd_config()
    absorption = replace(
        base.absorption,
        min_aggressive_volume=params["min_aggressive_volume"],
        max_price_displacement_ticks=params["absorption_max_price_displacement_ticks"],
        big_trade_filter=params["big_trade_filter"],
    )
    initiative = replace(
        base.initiative,
        min_delta_threshold=params["min_delta_threshold"],
        min_price_displacement_ticks=params["initiative_min_price_displacement_ticks"],
    )
    exhaustion = replace(base.exhaustion, volume_decline_pct=params["volume_decline_pct"])
    divergence = replace(
        base.divergence,
        lookback_bars=params["lookback_bars"],
        delta_failure_pct=params["delta_failure_pct"],
    )
    return replace(
        base, absorption=absorption, initiative=initiative, exhaustion=exhaustion, divergence=divergence
    )


def suggest_params(trial) -> dict:
    params = {}
    for name, (lo, hi, kind) in SEARCH_SPACE_BOUNDS.items():
        if kind == "float_log":
            params[name] = trial.suggest_float(name, lo, hi, log=True)
        elif kind == "float":
            params[name] = trial.suggest_float(name, lo, hi)
        elif kind == "int":
            params[name] = trial.suggest_int(name, lo, hi)
    return params

11 tunable parameters; baseline values all within their search ranges


In [13]:
_baseline_config = build_instrument_config(BASELINE_PARAMS)
assert _baseline_config.absorption.min_aggressive_volume == CONFIG.absorption.min_aggressive_volume
assert _baseline_config.initiative.min_delta_threshold == CONFIG.initiative.min_delta_threshold
assert _baseline_config.divergence.lookback_bars == CONFIG.divergence.lookback_bars
assert _baseline_config.volume_profile == CONFIG.volume_profile, "volume_profile is out of scope for tuning — must pass through untouched"
print("build_instrument_config(BASELINE_PARAMS) matches CONFIG on every tuned field")

build_instrument_config(BASELINE_PARAMS) matches CONFIG on every tuned field


## 4. Tunable Backtest Runner

**New code, not a reuse of baseline's `run_pipeline_replay`** — baseline's version
(notebook 01, §15) hardcodes the global `CONFIG` for all four detectors and only exposes
`SignalAggregator`'s 3 constructor kwargs as parameters. Since Task 3's search space
needs 8 additional detector-level fields, `run_backtest` below extends the same replay
loop to accept a full `InstrumentConfig` built fresh per trial by
`build_instrument_config` (Task 3) — the loop's *structure* mirrors baseline's, its
*signature* doesn't. `DAILY_PROFILES` (Task 1) stays global and untuned (volume-profile
params are out of scope — see Task 3), so it's reused as-is regardless of which slice of
`candles` is replayed.

In [14]:
from orderflow_system.analytics.delta import DeltaEngine
from orderflow_system.analytics.footprint import FootprintEngine
from orderflow_system.patterns.absorption import AbsorptionDetector
from orderflow_system.patterns.initiative import InitiativeDetector
from orderflow_system.patterns.exhaustion import ExhaustionDetector
from orderflow_system.patterns.divergence import DivergenceDetector
from orderflow_system.signals.aggregator import SignalAggregator


def run_backtest(
    candles: list,
    instrument_config: "InstrumentConfig",
    min_composite_score: float,
    signal_cooldown_seconds: float,
    price_proximity_pct: float,
) -> dict:
    """Replay `candles` through fresh detector/aggregator instances built from
    `instrument_config` + the three SignalAggregator kwargs. Mirrors baseline
    `run_pipeline_replay` (notebook 01 cell 44), extended to a full InstrumentConfig.
    Never mutates any global — every instance here is local to this call, which is what
    makes this function safe to call concurrently across Optuna trials/threads."""
    de = DeltaEngine(tick_size=TICK_SIZE)
    fe = FootprintEngine(tick_size=TICK_SIZE)
    absorption_d = AbsorptionDetector(instrument_config.absorption, tick_size=TICK_SIZE)
    initiative_d = InitiativeDetector(instrument_config.initiative, tick_size=TICK_SIZE)
    exhaustion_d = ExhaustionDetector(instrument_config.exhaustion)
    divergence_d = DivergenceDetector(instrument_config.divergence)
    framing = ProfileFramingEngine()
    agg = SignalAggregator(
        min_composite_score=min_composite_score,
        signal_cooldown_seconds=signal_cooldown_seconds,
        price_proximity_pct=price_proximity_pct,
    )

    raw_signals = []
    actions = []
    phase_trace = []
    recent = []
    current_day = None
    current_bias = None

    for c in candles:
        day = dt.datetime.fromtimestamp(c.timestamp_ms / 1000, tz=dt.timezone.utc).date()
        if day != current_day:
            prev_day, current_day = current_day, day
            if prev_day is not None and prev_day in DAILY_PROFILES:
                framing.add_profile(DAILY_PROFILES[prev_day])
                current_bias = framing.analyze(current_price=c.open)
                for level in current_bias.qualified_levels:
                    if level.strength >= 50:
                        agg.set_watching(SYMBOL, level, level.direction)

        d = de.compute_from_candle(c)
        fp_bar = fe.build_from_candle(c)
        signals = [
            s
            for s in (
                absorption_d.check_candle(c, fp_bar, d, c.close),
                initiative_d.check_candle(c, d, fp_bar),
                exhaustion_d.check_candle(c, d, de, fp_bar, recent),
                divergence_d.check_candle(c, de),
            )
            if s is not None
        ]
        raw_signals.extend(signals)

        for sig in signals:
            out = agg.process_signal(
                instrument=SYMBOL,
                signal=sig,
                bias=current_bias,
                current_price=c.close,
                recent_candles=recent[-5:],
            )
            if out is not None:
                actions.append((c.timestamp_ms, out))

        active = agg.get_active_trade(SYMBOL)
        phase_trace.append((c.timestamp_ms, active.phase.value if active else None))

        recent.append(c)
        if len(recent) > 20:
            recent = recent[-20:]

    return {"actions": actions, "raw_signals": raw_signals, "phase_trace": phase_trace}

In [15]:
_baseline_config = build_instrument_config(BASELINE_PARAMS)
_baseline_result = run_backtest(
    CANDLES,
    _baseline_config,
    min_composite_score=BASELINE_PARAMS["min_composite_score"],
    signal_cooldown_seconds=BASELINE_PARAMS["signal_cooldown_seconds"],
    price_proximity_pct=BASELINE_PARAMS["price_proximity_pct"],
)
assert len(_baseline_result["actions"]) == 0, (
    f"run_backtest with baseline params produced {len(_baseline_result['actions'])} actions — "
    "baseline notebook's own canonical run produced 0; if this now differs, run_backtest has "
    "diverged from run_pipeline_replay's logic (see Task 4 markdown)"
)
print("run_backtest(BASELINE_PARAMS) reproduces baseline's 0-trade canonical result — logic verified")

run_backtest(BASELINE_PARAMS) reproduces baseline's 0-trade canonical result — logic verified


## 5. Baseline Strategy Performance (Full History, No CV)

Shown for comparison only — this is the same canonical full-history run baseline
notebook §16 reports (`price_proximity_pct=0.002`, `min_composite_score=40.0`,
`signal_cooldown_seconds=30.0`, all BTCUSD runtime detector configs), reproduced here via
`run_backtest`/`BASELINE_PARAMS` instead of baseline's own `run_pipeline_replay` cell, so
this notebook's later "baseline vs tuned" comparison (Section 9) is computed through the
same code path for both sides. Expect 0 trades — see the Background section above.

In [16]:
_baseline_config = build_instrument_config(BASELINE_PARAMS)
_baseline_result = run_backtest(
    CANDLES,
    _baseline_config,
    min_composite_score=BASELINE_PARAMS["min_composite_score"],
    signal_cooldown_seconds=BASELINE_PARAMS["signal_cooldown_seconds"],
    price_proximity_pct=BASELINE_PARAMS["price_proximity_pct"],
)
BASELINE_LEDGER = simulate_trades_risk_based(_baseline_result["actions"], CANDLES)
BASELINE_METRICS = compute_financial_metrics(BASELINE_LEDGER, initial_capital=INITIAL_CAPITAL_USD)
BASELINE_METRICS["profit_factor"] = compute_profit_factor(BASELINE_LEDGER)
BASELINE_METRICS["final_equity"] = INITIAL_CAPITAL_USD + BASELINE_METRICS["total_pnl"]
BASELINE_METRICS["total_return_pct"] = BASELINE_METRICS["total_pnl"] / INITIAL_CAPITAL_USD * 100.0

print("Baseline strategy performance (full 32-day history, BTCUSD runtime params):")
for k, v in BASELINE_METRICS.items():
    print(f"  {k}: {v}")

if not BASELINE_LEDGER.empty:
    print("\nper-trade position sizing (capital_used == position_notional; no leverage):")
    print(BASELINE_LEDGER[["entry_time", "side", "position_notional", "capital_used", "dollar_risk", "stop_loss_pct", "quantity"]])

Baseline strategy performance (full 32-day history, BTCUSD runtime params):
  n_trades: 0
  total_pnl: 0.0
  win_rate_pct: 0.0
  sharpe_ratio: 0.0
  sortino_ratio: 0.0
  max_drawdown_pct: 0.0
  profit_factor: 0.0
  final_equity: 5000.0
  total_return_pct: 0.0


## 6. Purged/Embargoed CV Folds

Uses `RiskLabAI.backtest.validation.purged_kfold.PurgedKFold` (see Background section
above for why `times` is point-in-time and why `train_idx` is additionally filtered to
`< test_idx.min()`). `PurgedKFold.split(data)` only needs `data` to share `times`'s
index — no labels (`y`) are constructed or passed, matching the spec's "labels are not
required" note for a rule-based strategy.

Each fold's `context_idx` (causal warm-up) plus `test_idx` (scored window) together are
always a single contiguous prefix of `CANDLES` starting at index 0 — `run_backtest`
replays that whole prefix (state must build up from the start of a trading day/session
for `ProfileFramingEngine` and the rolling-window detectors to behave as the live system
would), and only actions whose candle falls in `[score_start_idx, test_idx.max()]` are
scored. `score_start_idx` adds an embargo buffer *inside* the test window itself (not
just what `PurgedKFold` purges from `train_idx`) so newly-warmed detector state right at
the fold boundary isn't scored.

In [17]:
import numpy as np
from RiskLabAI.backtest.validation.purged_kfold import PurgedKFold

N_CV_FOLDS = 5
EMBARGO_FRACTION = 0.02  # 2% of the dataset, purged after each test block + used as the in-test-window embargo buffer

CANDLE_INDEX_BY_TS = {c.timestamp_ms: i for i, c in enumerate(CANDLES)}
assert len(CANDLE_INDEX_BY_TS) == len(CANDLES), "duplicate candle timestamps -- CandleBuilder invariant violated"


def build_cv_folds(candles: list, n_splits: int = N_CV_FOLDS, embargo: float = EMBARGO_FRACTION) -> list[dict]:
    times_values = pd.Series([c.timestamp_ms for c in candles])
    times = pd.Series(times_values.values, index=times_values.values)  # point-in-time info range
    dummy = pd.DataFrame(index=times.index)

    splitter = PurgedKFold(n_splits=n_splits, times=times, embargo=embargo)
    embargo_candles = int(len(candles) * embargo)

    folds = []
    for train_idx, test_idx in splitter.split(dummy):
        causal_train_idx = train_idx[train_idx < test_idx.min()]
        folds.append({
            "context_idx": causal_train_idx,
            "test_idx": test_idx,
            "score_start_idx": int(test_idx.min()) + embargo_candles,
        })
    return folds


CV_FOLDS = build_cv_folds(CANDLES)

for i, fold in enumerate(CV_FOLDS):
    test_start_ts = CANDLES[int(fold["test_idx"].min())].timestamp_ms
    test_end_ts = CANDLES[int(fold["test_idx"].max())].timestamp_ms
    print(
        f"fold {i}: context={len(fold['context_idx']):,} candles, "
        f"test={len(fold['test_idx']):,} candles "
        f"({pd.Timestamp(test_start_ts, unit='ms', tz='UTC').date()} .. "
        f"{pd.Timestamp(test_end_ts, unit='ms', tz='UTC').date()}), "
        f"embargo trims first {fold['score_start_idx'] - int(fold['test_idx'].min())} test candles from scoring"
    )

assert len(CV_FOLDS) == N_CV_FOLDS
for fold in CV_FOLDS:
    assert fold["test_idx"].max() < len(CANDLES)
    assert fold["score_start_idx"] <= int(fold["test_idx"].max()) + 1, "embargo buffer consumed the entire test fold -- shrink EMBARGO_FRACTION or grow N_CV_FOLDS"
print(f"{N_CV_FOLDS} purged/embargoed folds built over {len(CANDLES):,} candles")

fold 0: context=0 candles, test=6,021 candles (2026-06-26 .. 2026-06-30), embargo trims first 602 test candles from scoring
fold 1: context=6,021 candles, test=6,021 candles (2026-06-30 .. 2026-07-04), embargo trims first 602 test candles from scoring
fold 2: context=12,042 candles, test=6,020 candles (2026-07-04 .. 2026-07-09), embargo trims first 602 test candles from scoring
fold 3: context=18,062 candles, test=6,020 candles (2026-07-09 .. 2026-07-13), embargo trims first 602 test candles from scoring
fold 4: context=24,082 candles, test=6,020 candles (2026-07-13 .. 2026-07-30), embargo trims first 602 test candles from scoring
5 purged/embargoed folds built over 30,102 candles


## 7. Fold Evaluator

`evaluate_fold` runs one fold's causal-prefix replay through `run_backtest`, keeps only
the actions whose candle index falls in `[score_start_idx, test_idx.max()]`, and turns
those into a ledger + metrics via Task 3's `simulate_trades_risk_based`/
`compute_financial_metrics`.

Two guards keep the objective well-defined everywhere in the search space, per the
Background section's warning that most parameter regions produce ~0 trades:

- **Trade-count floor** (`MIN_TRADES_PER_FOLD`): a fold with too few trades to compute a
  meaningful Sharpe is scored `PENALTY_SHARPE` instead of 0/NaN — 0 trades should read to
  Optuna as *worse* than a bad but real result, not neutral.
- **Blown-account guard**: unlike baseline's fixed notional, Task 3's sizing engine
  already makes "never exceed available capital" structural (`position_notional` is
  clamped to `equity_before` on every single trade — it cannot be violated by
  construction, so there is nothing left to check post-hoc here). What *can* still happen
  — and is worth a separate guard — is a fold where a string of losses (amplified by the
  no-intrabar-stop caveat in Task 3) drives `equity_after` to zero or below at some point.
  That parameter region is scored `PENALTY_SHARPE` too: a strategy that blows up its
  account partway through a fold isn't a valid "no leverage, capital-constrained" result
  regardless of what its Sharpe looks like on the trades that happened before the blowup.

In [18]:
MIN_TRADES_PER_FOLD = 2
PENALTY_SHARPE = -5.0


def evaluate_fold(fold: dict, candles: list, params: dict) -> dict:
    context_idx, test_idx = fold["context_idx"], fold["test_idx"]
    score_start_idx = fold["score_start_idx"]
    test_end_idx = int(test_idx.max())

    replay_candles = candles[: test_end_idx + 1]  # causal prefix (context) + test, contiguous from dataset start
    instrument_config = build_instrument_config(params)
    result = run_backtest(
        replay_candles,
        instrument_config,
        min_composite_score=params["min_composite_score"],
        signal_cooldown_seconds=params["signal_cooldown_seconds"],
        price_proximity_pct=params["price_proximity_pct"],
    )

    scored_actions = [
        (ts, action)
        for ts, action in result["actions"]
        if score_start_idx <= CANDLE_INDEX_BY_TS.get(ts, -1) <= test_end_idx
    ]
    ledger = simulate_trades_risk_based(scored_actions, replay_candles)
    metrics = compute_financial_metrics(ledger, initial_capital=INITIAL_CAPITAL_USD)
    metrics["profit_factor"] = compute_profit_factor(ledger)

    account_blown = bool((ledger["equity_after"] <= 0.0).any()) if not ledger.empty else False
    penalized = metrics["n_trades"] < MIN_TRADES_PER_FOLD or account_blown

    return {
        **metrics,
        "fold_sharpe": PENALTY_SHARPE if penalized else metrics["sharpe_ratio"],
        "account_blown": account_blown,
        "penalized": penalized,
        "n_context_candles": int(len(context_idx)),
        "n_test_candles": int(len(test_idx)),
        "n_scored_candles": test_end_idx - score_start_idx + 1,
        "avg_position_notional": float(ledger["position_notional"].mean()) if not ledger.empty else 0.0,
        "avg_dollar_risk": float(ledger["dollar_risk"].mean()) if not ledger.empty else 0.0,
    }

In [19]:
_fold0_result = evaluate_fold(CV_FOLDS[0], CANDLES, BASELINE_PARAMS)
print("fold 0 with BASELINE_PARAMS:", _fold0_result)
assert not _fold0_result["account_blown"], "baseline params should never blow the account -- baseline produces 0 trades, which trivially can't"
assert _fold0_result["penalized"], "baseline params produce ~0 trades across the whole 32-day sample -- fold 0 should hit the trade-count floor"
print("evaluate_fold penalizes baseline's known near-zero-trade behavior as expected")

fold 0 with BASELINE_PARAMS: {'n_trades': 0, 'total_pnl': 0.0, 'win_rate_pct': 0.0, 'sharpe_ratio': 0.0, 'sortino_ratio': 0.0, 'max_drawdown_pct': 0.0, 'profit_factor': 0.0, 'fold_sharpe': -5.0, 'account_blown': False, 'penalized': True, 'n_context_candles': 0, 'n_test_candles': 6021, 'n_scored_candles': 5419, 'avg_position_notional': 0.0, 'avg_dollar_risk': 0.0}
evaluate_fold penalizes baseline's known near-zero-trade behavior as expected


## 8. Optuna Objective & Persisted Study

`objective(trial)` samples one parameter set (Task 3's `suggest_params`), evaluates it
on every purged/embargoed fold (Task 7's `evaluate_fold`), and returns the mean
out-of-sample fold Sharpe -- never a single full-history in-sample Sharpe. Per-fold
detail (trade counts, penalties, individual Sharpes) is stashed on
`trial.user_attrs["fold_results"]` so Section 9's summary can report it without
re-running anything.

**Reproducibility:** the backtest itself has no randomness -- the same `params` dict
always produces the same trades on the same `CANDLES`. `TPESampler(seed=42)` fixes the
*sampler's* randomness. **Parallelism:** `study.optimize(..., n_jobs=N)` runs trials in
threads; each trial builds entirely fresh `InstrumentConfig`/detector/`SignalAggregator`
instances inside `run_backtest`/`build_instrument_config` (Task 4/5) and never mutates a
shared global, so concurrent trials cannot corrupt each other's state -- this is what
"safe" parallelism means here. Because it's thread- not process-based, Python's GIL caps
the real speedup for this CPU-bound loop; a process-pool alternative would need to pickle
`CANDLES` across process boundaries and is left out of scope for this first cut (noted
here, not silently assumed).

In [20]:
import optuna

STUDY_DB_PATH = Path(".optuna_studies/orderflow_cv_tuning.db")  # relative to THIS notebook's own
# directory -- see Task 2's CV_CACHE_DIR note; resolves to notebook/.optuna_studies/... regardless
# of how the notebook is launched, since nbconvert/Jupyter set cwd to the .ipynb's own directory
STUDY_DB_PATH.parent.mkdir(parents=True, exist_ok=True)
STUDY_NAME = "orderflow_cv_tuning_btcusd"


def objective(trial: "optuna.Trial") -> float:
    params = suggest_params(trial)
    fold_results = [evaluate_fold(fold, CANDLES, params) for fold in CV_FOLDS]
    fold_sharpes = [f["fold_sharpe"] for f in fold_results]
    mean_cv_sharpe = float(np.mean(fold_sharpes))

    trial.set_user_attr("fold_results", fold_results)
    trial.set_user_attr("cv_sharpe_mean", mean_cv_sharpe)
    trial.set_user_attr("cv_sharpe_std", float(np.std(fold_sharpes)))
    trial.set_user_attr("total_trades", int(sum(f["n_trades"] for f in fold_results)))
    trial.set_user_attr("n_bars_used", len(CANDLES))

    return mean_cv_sharpe


# n_jobs>1 (Section 9) means multiple threads commit to this SQLite file concurrently.
# The bare "sqlite:///..." URL uses Python's sqlite3 default 5s busy-timeout, which is too
# short under real contention -- confirmed by a genuine "database is locked" ->
# optuna.exceptions.StorageInternalError crash during development with n_jobs=4. Raising
# the connection timeout (how long a thread waits for the write lock before giving up)
# is Optuna's own documented fix for this exact failure mode.
_storage = optuna.storages.RDBStorage(
    url=f"sqlite:///{STUDY_DB_PATH}",
    engine_kwargs={"connect_args": {"timeout": 60}},
)

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=_storage,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    load_if_exists=True,
)
print(f"study '{STUDY_NAME}' loaded from {STUDY_DB_PATH} with {len(study.trials)} historical trial(s)")

/Users/bobet/Documents/Code-Repository/Trading/orderflow-analysis-pro/.venv_orderflow/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[I 2026-09-12 21:30:05,010] Using an existing study with name 'orderflow_cv_tuning_btcusd' instead of creating a new one.


study 'orderflow_cv_tuning_btcusd' loaded from .optuna_studies/orderflow_cv_tuning.db with 92 historical trial(s)


## 9. Run the Study

`N_TRIALS` new trials per run, `N_JOBS` of them concurrently (see Task 8's parallelism
note). Because `load_if_exists=True` was used when creating the study, re-running this
cell in a later session adds `N_TRIALS` *more* trials on top of whatever's already in
the SQLite file -- it does not restart from scratch. Runtime scales with `N_CV_FOLDS`
and the size of `CANDLES` (each trial replays a causal-prefix growing across
`N_CV_FOLDS` folds, roughly `(N_CV_FOLDS + 1) / 2` full-history-equivalent replays per
trial) -- 30 trials is a reasonable first pass; raise `N_TRIALS` for a deeper search once
this runs cleanly end-to-end.

In [21]:
N_TRIALS = 30
N_JOBS = 4

study.optimize(objective, n_trials=N_TRIALS, n_jobs=N_JOBS, show_progress_bar=True)

_completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
best = study.best_trial

print("=" * 60)
print("BEST RESULT SUMMARY")
print("=" * 60)
print(f"best trial number:       {best.number}")
print(f"best mean CV Sharpe:     {best.value:.4f}")
print(f"best params:")
for k, v in best.params.items():
    print(f"    {k}: {v}")
print(f"total completed trials:  {len(_completed)}")
print(f"historical bars used:    {best.user_attrs['n_bars_used']:,}")
print(f"total trades (best):     {best.user_attrs['total_trades']}")
print(f"CV sharpe mean/std:      {best.user_attrs['cv_sharpe_mean']:.4f} / {best.user_attrs['cv_sharpe_std']:.4f}")
print("per-fold stats (best trial):")
for i, fr in enumerate(best.user_attrs["fold_results"]):
    print(
        f"    fold {i}: sharpe={fr['fold_sharpe']:.4f}  n_trades={fr['n_trades']}  "
        f"win_rate={fr['win_rate_pct']:.1f}%  profit_factor={fr['profit_factor']:.2f}  "
        f"max_dd={fr['max_drawdown_pct']:.2f}%  penalized={fr['penalized']}"
    )

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:59<?, ?it/s]

Best trial: 0. Best value: -5:   0%|          | 0/30 [00:59<?, ?it/s]

Best trial: 0. Best value: -5:   3%|▎         | 1/30 [00:59<28:49, 59.65s/it]

Best trial: 0. Best value: -5:   3%|▎         | 1/30 [00:59<28:49, 59.65s/it]

...

Best trial: 0. Best value: -5:  97%|█████████▋| 29/30 [07:27<00:14, 14.04s/it]

Best trial: 0. Best value: -5:  97%|█████████▋| 29/30 [07:27<00:14, 14.04s/it]

Best trial: 0. Best value: -5:  97%|█████████▋| 29/30 [07:27<00:14, 14.04s/it]

Best trial: 0. Best value: -5: 100%|██████████| 30/30 [07:27<00:00, 14.91s/it]

[I 2026-09-12 21:37:32,402] Trial 121 finished with value: -5.0 and parameters: {'price_proximity_pct': 0.00814445379377977, 'min_composite_score': 60.6381630823232, 'signal_cooldown_seconds': 75.08671249104289, 'min_aggressive_volume': 91.12087879499761, 'absorption_max_price_displacement_ticks': 2.1773197602105103, 'big_trade_filter': 18.297866685556045, 'min_delta_threshold': 55.619020168731495, 'initiative_min_price_displacement_ticks': 3.2204816843778725, 'volume_decline_pct': 0.16626797410944993, 'lookback_bars': 25, 'delta_failure_pct': 0.74941250696074}. Best is trial 0 with value: -5.0.
[I 2026-09-12 21:37:32,413] Trial 120 finished with value: -5.0 and parameters: {'price_proximity_pct': 0.015429165808243458, 'min_composite_score': 15.198972292794197, 'signal_cooldown_seconds': 74.60054670599074, 'min_aggressive_volume': 73.2568551855547, 'absorption_max_price_displacement_ticks': 3.1091467416841057, 'big_trade_filter': 13.057348338938358, 'min_delta_threshold': 15.2842448220

In [22]:
_trials_before = len(study.trials)
study.optimize(objective, n_trials=5, n_jobs=N_JOBS)
_trials_after = len(study.trials)
assert _trials_after == _trials_before + 5, (
    f"expected {_trials_before + 5} trials after a second optimize() call, got {_trials_after} -- "
    "study persistence (load_if_exists=True) is not accumulating trials as expected"
)
print(f"persistence verified: {_trials_before} -> {_trials_after} trials across two optimize() calls")

[I 2026-09-12 21:38:32,757] Trial 125 finished with value: -5.0 and parameters: {'price_proximity_pct': 0.01936682616525586, 'min_composite_score': 33.39024525866593, 'signal_cooldown_seconds': 20.601101420834002, 'min_aggressive_volume': 97.04683961371452, 'absorption_max_price_displacement_ticks': 4.574627574898298, 'big_trade_filter': 5.659701527170258, 'min_delta_threshold': 60.22695230487919, 'initiative_min_price_displacement_ticks': 1.0462977253827774, 'volume_decline_pct': 0.214692181109669, 'lookback_bars': 6, 'delta_failure_pct': 0.507112582587537}. Best is trial 0 with value: -5.0.


[I 2026-09-12 21:38:32,764] Trial 122 finished with value: -5.0 and parameters: {'price_proximity_pct': 0.019302419001388536, 'min_composite_score': 33.217496400108416, 'signal_cooldown_seconds': 96.53836735453251, 'min_aggressive_volume': 98.86203208862369, 'absorption_max_price_displacement_ticks': 4.564883828225627, 'big_trade_filter': 12.555922641308445, 'min_delta_threshold': 45.24954198129473, 'initiative_min_price_displacement_ticks': 5.606498206990957, 'volume_decline_pct': 0.28294761646372907, 'lookback_bars': 14, 'delta_failure_pct': 0.5093270259233424}. Best is trial 0 with value: -5.0.


[I 2026-09-12 21:38:32,775] Trial 123 finished with value: -5.0 and parameters: {'price_proximity_pct': 0.01888690503152629, 'min_composite_score': 32.861235434265055, 'signal_cooldown_seconds': 55.498918974049836, 'min_aggressive_volume': 97.70487127252377, 'absorption_max_price_displacement_ticks': 5.405970156695105, 'big_trade_filter': 4.69465012706428, 'min_delta_threshold': 44.86503008266164, 'initiative_min_price_displacement_ticks': 5.608559770041666, 'volume_decline_pct': 0.28259961709215653, 'lookback_bars': 14, 'delta_failure_pct': 0.5050010027777286}. Best is trial 0 with value: -5.0.


[I 2026-09-12 21:38:34,358] Trial 124 finished with value: -5.0 and parameters: {'price_proximity_pct': 0.0178961881122479, 'min_composite_score': 31.616974208845477, 'signal_cooldown_seconds': 88.13675454778785, 'min_aggressive_volume': 98.22645937740474, 'absorption_max_price_displacement_ticks': 5.443662939342807, 'big_trade_filter': 4.346849324510676, 'min_delta_threshold': 45.25777446607505, 'initiative_min_price_displacement_ticks': 4.890682554634141, 'volume_decline_pct': 0.2887658785892832, 'lookback_bars': 14, 'delta_failure_pct': 0.5021432351409258}. Best is trial 0 with value: -5.0.


[I 2026-09-12 21:38:45,843] Trial 126 finished with value: -5.0 and parameters: {'price_proximity_pct': 0.01657857185636651, 'min_composite_score': 31.274928085214466, 'signal_cooldown_seconds': 87.92214625463447, 'min_aggressive_volume': 85.06397685530155, 'absorption_max_price_displacement_ticks': 5.534427098469891, 'big_trade_filter': 9.617362332077064, 'min_delta_threshold': 57.78694982463847, 'initiative_min_price_displacement_ticks': 7.127419939921787, 'volume_decline_pct': 0.499734158751732, 'lookback_bars': 22, 'delta_failure_pct': 0.5250578422753384}. Best is trial 0 with value: -5.0.


persistence verified: 122 -> 127 trials across two optimize() calls


## 10. Baseline vs. Tuned — Final Comparison

Two different things are shown side by side and must not be conflated: the tuned
row's **full-history** columns are an in-sample display (same convention as the
baseline row and baseline notebook §16, for a like-for-like comparison of what each
parameter set does over the *entire* 32-day sample) -- the tuned row's **CV Sharpe**
column is the actual out-of-sample number Optuna optimized (Section 8's `mean_cv_sharpe`),
computed only from held-out fold windows. Only the CV Sharpe column is evidence this
parameter set generalizes; the full-history columns are diagnostic context.

**Honest disclosure of this run's result:** every trial completed in Section 9 was
fully penalized (`mean CV Sharpe = -5.0` for every one, `best.value == -5.0`) -- no
sampled parameter region within this pass escaped the trade-count floor across all 5
folds. Per the Background section, this strategy has a real, unfixed repo state-machine
dead-end (`ABSORPTION_DETECTED` has no routing branch) that makes most of the search
space untradeable; a search of this size is not nearly enough to sample an
11-dimensional space densely enough to reliably land in whatever narrow region (if any)
escapes it. This is reported as found, not loosened away: `MIN_TRADES_PER_FOLD`,
`PENALTY_SHARPE`, and the search ranges are unchanged from Sections 3 and 7. The
comparison below is therefore between baseline's own 0-trade canonical result and this
run's *best available* (still-penalized) trial -- both rows are 0-or-near-0-trade
results, and the honest reading is "this pass didn't find a working region," not
"tuning improved the strategy." Re-running Section 9 with a larger `N_TRIALS` (raised
in that cell) is how a deeper search would be attempted; nothing about that requires
changing this section's code.


In [23]:
_tuned_config = build_instrument_config(best.params)
_tuned_result = run_backtest(
    CANDLES,
    _tuned_config,
    min_composite_score=best.params["min_composite_score"],
    signal_cooldown_seconds=best.params["signal_cooldown_seconds"],
    price_proximity_pct=best.params["price_proximity_pct"],
)
_tuned_ledger = simulate_trades_risk_based(_tuned_result["actions"], CANDLES)
_tuned_metrics = compute_financial_metrics(_tuned_ledger, initial_capital=INITIAL_CAPITAL_USD)
_tuned_metrics["profit_factor"] = compute_profit_factor(_tuned_ledger)
_tuned_metrics["final_equity"] = INITIAL_CAPITAL_USD + _tuned_metrics["total_pnl"]
_tuned_metrics["total_return_pct"] = _tuned_metrics["total_pnl"] / INITIAL_CAPITAL_USD * 100.0

COMPARISON = pd.DataFrame(
    [
        {
            "run": "baseline (full-history)",
            "n_trades": BASELINE_METRICS["n_trades"],
            "total_return_pct": BASELINE_METRICS["total_return_pct"],
            "final_equity": BASELINE_METRICS["final_equity"],
            "sharpe_ratio (full-history)": BASELINE_METRICS["sharpe_ratio"],
            "max_drawdown_pct": BASELINE_METRICS["max_drawdown_pct"],
            "win_rate_pct": BASELINE_METRICS["win_rate_pct"],
            "profit_factor": BASELINE_METRICS["profit_factor"],
            "avg_position_notional": float(BASELINE_LEDGER["position_notional"].mean()) if not BASELINE_LEDGER.empty else 0.0,
            "avg_dollar_risk": float(BASELINE_LEDGER["dollar_risk"].mean()) if not BASELINE_LEDGER.empty else 0.0,
            "CV Sharpe (mean ± std)": None,
        },
        {
            "run": "tuned (full-history)",
            "n_trades": _tuned_metrics["n_trades"],
            "total_return_pct": _tuned_metrics["total_return_pct"],
            "final_equity": _tuned_metrics["final_equity"],
            "sharpe_ratio (full-history)": _tuned_metrics["sharpe_ratio"],
            "max_drawdown_pct": _tuned_metrics["max_drawdown_pct"],
            "win_rate_pct": _tuned_metrics["win_rate_pct"],
            "profit_factor": _tuned_metrics["profit_factor"],
            "avg_position_notional": float(_tuned_ledger["position_notional"].mean()) if not _tuned_ledger.empty else 0.0,
            "avg_dollar_risk": float(_tuned_ledger["dollar_risk"].mean()) if not _tuned_ledger.empty else 0.0,
            "CV Sharpe (mean ± std)": f"{best.user_attrs['cv_sharpe_mean']:.4f} ± {best.user_attrs['cv_sharpe_std']:.4f}",
        },
    ]
)
COMPARISON

,run,n_trades,total_return_pct,final_equity,sharpe_ratio (full-history),max_drawdown_pct,win_rate_pct,profit_factor,avg_position_notional,avg_dollar_risk,CV Sharpe (mean ± std)
0,baseline (full-history),0,0.000000,5000.000000,0.00000,0.0,0.0,0.0,0.0,0.000000,NaN
1,tuned (full-history),1,8.543536,5427.176778,3.22933,0.0,100.0,inf,5000.0,5.537842,-5.0000 ± 0.0000


In [24]:
import optuna.visualization as vis

_history_fig = vis.plot_optimization_history(study)
_history_fig.show()

In [25]:
_completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
_pruned_or_failed = [t for t in study.trials if t.state != optuna.trial.TrialState.COMPLETE]

print("=" * 60)
print("STUDY METADATA")
print("=" * 60)
print(f"study name:              {STUDY_NAME}")
print(f"SQLite database path:    {STUDY_DB_PATH.resolve()}")
print(f"total trials recorded:   {len(study.trials)}")
print(f"completed trials:        {len(_completed)}")
print(f"non-completed trials:    {len(_pruned_or_failed)}")
print(f"historical bars used:    {len(CANDLES):,}")
print(f"best trial so far:       #{study.best_trial.number} (mean CV Sharpe = {study.best_value:.4f})")

STUDY METADATA
study name:              orderflow_cv_tuning_btcusd
SQLite database path:    /Users/bobet/Documents/Code-Repository/Trading/orderflow-analysis-pro/.claude/worktrees/cross-validation-tuning/notebook/.optuna_studies/orderflow_cv_tuning.db
total trials recorded:   127
completed trials:        126
non-completed trials:    1
historical bars used:    30,102
best trial so far:       #0 (mean CV Sharpe = -5.0000)
